In [16]:
"""
graph_test.py

Funciones de prueba para construir un grafo a partir de un GeoDataFrame usando k-nearest
neighbors y calcular distancias mínimas desde múltiples fuentes con Dijkstra.

Cómo usar:
  python graph_test.py    # intenta localizar dataset_real (dataset_final.gpkg) en rutas probables

La prueba carga un GeoDataFrame real si lo encuentra, construye el grafo y calcula distancias
desde un subconjunto de nodos "estaciones". Imprime estadísticas básicas y algunas comprobaciones.
"""
from pathlib import Path
from typing import List, Tuple, Dict

import geopandas as gpd
import networkx as nx
import numpy as np
from sklearn.neighbors import NearestNeighbors
from pyproj import Geod, CRS


def build_graph_from_gdf(gdf: gpd.GeoDataFrame, k: int = 4) -> nx.Graph:
    """Construye un grafo no dirigido a partir de `gdf` conectando cada nodo a sus k vecinos.

    Se asume que `gdf` contiene una columna `osmid`. Si no tiene columnas `x`/`y`, se usan
    las coordenadas del centroide de la geometría.

    Los atributos añadidos a las aristas: 'length' (distancia euclidiana entre puntos).
    """
    # Asegurar columnas x,y
    if 'x' not in gdf.columns or 'y' not in gdf.columns:
        centroids = gdf.geometry.centroid
        gdf = gdf.copy()
        gdf['x'] = centroids.x
        gdf['y'] = centroids.y

    coords = np.array(list(zip(gdf['x'], gdf['y'])))

    # Build neighbors
    nbrs = NearestNeighbors(n_neighbors=min(k + 1, len(gdf)), algorithm='auto').fit(coords)
    distances, indices = nbrs.kneighbors(coords)

    G = nx.Graph()

    # Añadir nodos
    for idx, row in gdf.iterrows():
        osmid = row.get('osmid', idx)
        G.add_node(osmid, pos=(row['x'], row['y']))

    # Asegurar columna osmid (si no existe, usar el índice como identificador)
    if 'osmid' not in gdf.columns:
        gdf = gdf.copy()
        gdf['osmid'] = list(gdf.index)

    # Añadir aristas siguiendo el snippet original
    for i, osmid in enumerate(gdf.get('osmid', gdf.index)):
        for j in range(1, min(k + 1, indices.shape[1])):  # índice 0 es el mismo nodo
            vecino_idx = indices[i, j]
            vecino_osmid = gdf['osmid'].iloc[vecino_idx]
            dist = float(distances[i, j])
            # Añadir arista (NetworkX manejará duplicados)
            G.add_edge(osmid, vecino_osmid, length=dist)

    return G


def compute_multi_source_distances(G: nx.Graph, estaciones_osmid: List[int]) -> Dict[int, float]:
    """Calcula distancias mínimas desde múltiples nodos fuente usando Dijkstra (peso 'length')."""
    # Asegurarse de que las estaciones existen en el grafo
    estaciones_presentes = [s for s in estaciones_osmid if s in G]
    if len(estaciones_presentes) == 0:
        raise ValueError("Ninguna de las estaciones dadas existe en el grafo.")

    distancias = nx.multi_source_dijkstra_path_length(G, estaciones_presentes, weight='length')

    return distancias


def find_dataset_candidates() -> List[Path]:
    """Devuelve una lista de rutas candidatas donde puede estar `dataset_final.gpkg` en este repo."""
    # En un notebook `__file__` no está definido; usar cwd como fallback
    try:
        here = Path(__file__).resolve().parent
    except NameError:
        here = Path.cwd()
    candidates = [
        here / 'dataset_final.gpkg',
        here.parent.parent / 'unificacion_puntos_SJ' / 'unificacion_puntos_SJ' / 'dataset_final.gpkg',
        here.parent.parent.parent / 'unificacion_puntos_SJ' / 'unificacion_puntos_SJ' / 'dataset_final.gpkg',
        Path.cwd() / 'dataset_final.gpkg'
    ]
    # Filtrar existentes
    return [p for p in candidates if p.exists()]


def test_graph_distances(dataset_path: Path = None, k: int = 4, n_estaciones: int = 3) -> Tuple[nx.Graph, Dict[int, float]]:
    """Carga el dataset (si se proporciona o encontrado automáticamente), construye el grafo y calcula distancias.

    Devuelve el grafo y el diccionario de distancias.
    También realiza algunas comprobaciones básicas y prints de diagnóstico.
    """
    # Localizar dataset si no fue dado
    if dataset_path is None:
        candidates = find_dataset_candidates()
        if not candidates:
            raise FileNotFoundError('No se encontró dataset_final.gpkg en rutas candidatas. Pasa dataset_path explícitamente.')
        dataset_path = candidates[0]

    print(f"Cargando dataset desde: {dataset_path}")
    gdf = gpd.read_file(str(dataset_path))

    print(f"Registros leídos: {len(gdf)}")

    # Guardar CRS original
    orig_crs = CRS.from_user_input(gdf.crs) if gdf.crs is not None else None

    # Si no está proyectado, proyectar a EPSG:3857 para distancias en metros
    if orig_crs is None or not orig_crs.is_projected:
        gdf_proj = gdf.to_crs(epsg=3857)
        projected = True
    else:
        gdf_proj = gdf.copy()
        projected = False

    # Añadir columnas x,y en gdf_proj para construir el grafo
    cent = gdf_proj.geometry.centroid
    gdf_proj = gdf_proj.copy()
    gdf_proj['x'] = cent.x
    gdf_proj['y'] = cent.y

    # Construir grafo en coordenadas proyectadas (metros)
    G = build_graph_from_gdf(gdf_proj, k=k)
    print(f"Grafo construido (proyectado): nodos={G.number_of_nodes()}, aristas={G.number_of_edges()}")

    # Asegurar columna osmid en el gdf original
    if 'osmid' not in gdf.columns:
        gdf = gdf.copy()
        gdf['osmid'] = list(gdf.index)

    # Seleccionar estaciones ejemplo: tomar n_estaciones equiespaciadas (en la lista de osmid)
    osmid_list = list(gdf['osmid'])
    indices = np.linspace(0, len(osmid_list) - 1, num=min(n_estaciones, len(osmid_list)), dtype=int)
    estaciones_osmid = [osmid_list[i] for i in indices]
    print(f"Estaciones de ejemplo (osmid): {estaciones_osmid}")

    # Calcular distancias (en metros si usamos proyección EPSG:3857)
    distancias = compute_multi_source_distances(G, estaciones_osmid)

    # Mapear las distancias calculadas de vuelta al GeoDataFrame original
    # Si falta algún nodo en el dict (nodo no alcanzable), asignamos np.inf
    gdf = gdf.copy()
    gdf['dist_to_station_m'] = gdf['osmid'].map(lambda o: distancias.get(o, np.inf))

    # Estadísticas básicas (del dict de distancias)
    dist_values = np.array(list(distancias.values())) if len(distancias) > 0 else np.array([])
    if dist_values.size > 0:
        print(f"Distancias calculadas (m) para {len(dist_values)} nodos (de {G.number_of_nodes()}): min={dist_values.min():.2f}, mean={dist_values.mean():.2f}, max={dist_values.max():.2f}")
    else:
        print("No se calcularon distancias (grafo desconectado o estaciones vacías).")

    # Convertir las distancias (metros) a la unidad original del GeoDataFrame si era geográfica
    if orig_crs is not None and orig_crs.is_geographic:
        geod = Geod(ellps='WGS84')

        def meters_to_deg_lat(lat, lon, meters):
            if meters is None or np.isinf(meters):
                return np.nan
            # Avanzar hacia el norte (azimut 0) la distancia en metros y calcular la diferencia en grados de latitud
            lon2, lat2, _ = geod.fwd(lon, lat, 0, float(meters))
            return abs(lat2 - lat)

        # Centroides en lon/lat (original)
        cent_orig = gdf.geometry.centroid
        gdf['lon'] = cent_orig.x
        gdf['lat'] = cent_orig.y
        gdf['dist_to_station_deg_lat'] = gdf.apply(lambda r: meters_to_deg_lat(r['lat'], r['lon'], r['dist_to_station_m']), axis=1)
        print("Se añadió columna 'dist_to_station_deg_lat' (equivalente en grados de latitud aproximados).")
    else:
        # Si el CRS original está en metros (proyectado), la unidad ya es metros
        gdf['dist_to_station_orig_units'] = gdf['dist_to_station_m']
        print("CRS original proyectado: la distancia ya está en las mismas unidades (probablemente metros).")

    # Mostrar primeras 10 distancias por osmid en el GeoDataFrame
    print("Primeras 10 filas con dist_to_station (m y original units):")
    cols_to_show = ['osmid', 'dist_to_station_m'] + (['dist_to_station_deg_lat'] if 'dist_to_station_deg_lat' in gdf.columns else ['dist_to_station_orig_units'])
    print(gdf[cols_to_show].head(10))

    # (Opcional) Guardar un geopackage nuevo con la columna de distancia añadida
    try:
        out_path = dataset_path.with_name(dataset_path.stem + '_with_distances.gpkg')
        gdf.to_file(str(out_path), layer='nodos_unificados_dist', driver='GPKG')
        print(f"Archivo con distancias guardado en: {out_path}")
    except Exception:
        print("No se pudo guardar el geopackage con distancias (probablemente driver faltante).")

    return G, distancias


if __name__ == '__main__':
    import argparse

    parser = argparse.ArgumentParser(description='Test graph distances (kNN + multi-source Dijkstra)')
    parser.add_argument('--dataset', '-d', type=str, default=None, help='Ruta al dataset_final.gpkg')
    parser.add_argument('--k', type=int, default=4, help='Número de vecinos k para construir el grafo')
    parser.add_argument('--n_estaciones', type=int, default=3, help='Número de estaciones (fuentes) a usar en Dijkstra')

    # Usar parse_known_args para ignorar los argumentos que Jupyter inyecta (ej. --f=...)
    args, _ = parser.parse_known_args()

    dataset_arg = Path(args.dataset) if args.dataset else None

    try:
        G, dist = test_graph_distances(dataset_path=dataset_arg, k=args.k, n_estaciones=args.n_estaciones)
        print('\n✅ Prueba completada con éxito.')
    except Exception as e:
        print('\n❌ Error durante la prueba:')
        import traceback

        traceback.print_exc()


Cargando dataset desde: c:\Users\Franc\Desktop\Informatica\.CURSANDO\Int Comp - Inteligencia Computational - 2025\TP Final\Implementacion\algoritmo_NSGA_2\algoritmo_NSGA_2\dataset_final.gpkg
Registros leídos: 488
Grafo construido (proyectado): nodos=488, aristas=1239
Estaciones de ejemplo (osmid): [90858631, 90928361, 90795981]
Distancias calculadas (m) para 446 nodos (de 488): min=0.00, mean=38366.88, max=74916.83
CRS original proyectado: la distancia ya está en las mismas unidades (probablemente metros).
Primeras 10 filas con dist_to_station (m y original units):
        osmid  dist_to_station_m  dist_to_station_orig_units
0    90858631           0.000000                    0.000000
1    90858633         525.366076                  525.366076
2  8315283498         929.833998                  929.833998
3    90918359        9426.013841                 9426.013841
4    90876964       14403.682469                14403.682469
5    90955033       12663.946022                12663.946022
6